In [40]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = (
            SparkSession
                .builder
                .master("local[*]")
                .appName("notebook_teste")
                .config(
                    "spark.jars.packages",
                    "org.apache.hadoop:hadoop-aws:3.3.4,"
                    "com.amazonaws:aws-java-sdk-bundle:1.12.261,"
                    "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.7.1,"
                    "org.scala-lang:scala-library:2.13.12")
                # Configuração do Catálogo Iceberg
                .config("spark.sql.catalog.icebergtb", "org.apache.iceberg.spark.SparkCatalog")
                .config("spark.driver.extraJavaOptions", "-Dlog4j.configuration=log4j.properties") 
                .config("spark.log.level", "ERROR") 
                .config("spark.eventLog.enabled", "false") 
                # .config("spark.sql.catalog.icebergtb.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog")
                .config("spark.sql.catalog.icebergtb.type", "hadoop") 
                .config("spark.sql.catalog.icebergtb.warehouse", "s3a://iceberg/")
                .config("spark.sql.catalog.icebergtb.hadoop.fs.s3a.endpoint", "http://172.19.192.1:9000")
                .config("spark.sql.catalog.icebergtb.hadoop.fs.s3a.access.key", "minioadmin")
                .config("spark.sql.catalog.icebergtb.hadoop.fs.s3a.secret.key", "minioadmin")
                .config("spark.sql.catalog.icebergtb.hadoop.fs.s3a.path.style.access", "true")
                .config("spark.sql.catalog.icebergtb.hadoop.fs.s3a.connection.ssl.enabled", "false")
                .config("spark.executor.extraJavaOptions", "-Dio.netty.tryReflectionSetAccessible=true") 
                .config("spark.driver.extraJavaOptions", "-Dio.netty.tryReflectionSetAccessible=true") 
            .getOrCreate()
)

spark

In [41]:
dataframe = spark.sql(""" select 
                            row_number() over (order by storeid) as storekey,
                            *
                            from icebergtb.advworks_rdb.raw_dimstore  where personid is not null""")

In [42]:
dataframe = (dataframe
                .withColumn("businesstype", regexp_replace("businesstype", r"(?i)\bbm\b", "Bike Mall"))
                .withColumn("businesstype", regexp_replace("businesstype", r"(?i)\bos\b", "Online Store"))
                .withColumn("businesstype", regexp_replace("businesstype", r"(?i)\bbs\b", "Bike Store"))
                .withColumn("yearsinactivity", year(current_date()) - col("yearopened"))
)


In [43]:
dataframe = dataframe.withColumn(
                    "storesize",
                        when(col("annualrevenue") <= 30000.00, "Local Store")
                    .when((col("annualrevenue") > 30000.00) & (col("annualrevenue") < 80000.00), "Small Store")
                    .when((col("annualrevenue") > 80000.00) & (col("annualrevenue") < 100000.00), "Medium Store")
                    .when((col("annualrevenue") > 100000.00) & (col("annualrevenue") <= 150000.00), "Large Store")
                .otherwise("Global Store")
)
            

In [44]:
dataframe = dataframe.withColumn(
                        "brandExpertise",
                        when((col("brands") == "3"), "Senior")
                        .when((col("brands") >= 1) & (col("brands") <= 2), "Medium Knowledge")
                        .otherwise("Expert")
)

In [58]:
store = (dataframe.select(      "storekey",			"storeid",
                                "PersonID",			"CustomerID",
                                "AccountNumber", 	"storename",
                                "FirstName",		"LastName",
                                "AnnualSales",		"AnnualRevenue",
                                "BankName",			"businesstype",
                                "YearOpened",		"Specialty",
                                "SquareFeet",		"Brands",
                                "Internet",			"NumberEmployees",
                                "SalesPersonID",	"territoryid",
                                "continentname",	"regionname",
                                "CountryRegionCode","yearsinactivity",
                                "storesize",		"brandExpertise",	
                                "inserted_date",	"modified_date",
                                "eltversion"
                            ).withColumnRenamed("territoryid", "countryid")
                             .withColumnRenamed("personid", "clientid")
                             .withColumnRenamed("salespersonid", "vendorid")
)

